# How to use
1. Create huggingface account
2. create a read token and a read_write token
3. add it to secrets (key on left) as HF_TOKEN and HF_TOKEN_ALL
4. set `USE_TESSERA`, `TS_LENGTH`, `TS_INTERVAL`, `TRAIN_VAL_LABEL_SEL` and the repo names in the first code cell, run the setup cells, then uncomment the step you need
5. data will be saved

All logic lives in two files of the repo: `firepred/data/windows_master.py` (windows, zips, downloads) and `firepred/data/tessera_master.py` (TESSERA). This notebook only sets a few values and calls them; each call prints a short report.

**Hugging Face datasets** (defaults below; change any of them in the first code cell):
| constant | repo | contents |
|---|---|---|
| `SRC_REPO` | `SamuelWu318/ts-satfire` | raw GeoTIFFs + ROI CSVs |
| `WINDOW_REPO` | `SamuelWu318/ts-satfire-processed-<TS_LENGTH>-<TS_INTERVAL>` (derived) | windowed NPZ per fire + `data.zip`; use one repo per `TS_LENGTH` / `TS_INTERVAL` setting |
| `TESSERA_REPO` | `SamuelWu318/ts-tesserafire` | per-fire TESSERA embeddings + `tessera.zip` |

**Storage:** every download, cache, and temporary file stays on this Colab VM under `/content` (`WORK_DIR`). Colab's disk is wiped when the session ends, so finished data is uploaded to Hugging Face.

**TESSERA mode** (`USE_TESSERA = True`): PULL ZIP adds `tessera.zip` (embeddings + drop list) next to `data.zip` in its own `DATA_ROOT`, then removes the window archives of fires with no prior-year TESSERA (all 2017 fires). Window data is stored once, in `WINDOW_REPO`'s `data.zip`.

In [ ]:
USE_TESSERA = False  # True: training data gets per-fire TESSERA embeddings and drops 2017 fires

# Window settings (defaults in firepred/config/dataset.py). A fire needs TS_LENGTH + 1 days.
# The TS-SatFire paper's prediction benchmark uses 6 input days. Each setting has its own window repo.
TS_LENGTH = 6    # input days per window
TS_INTERVAL = 1   # days between window starts

# Train/val burned-area labels (TRAIN_VAL_LABEL_SEL in firepred/config/dataset.py).
#   1: accumulated active-fire (AF) detections only, no NIFC perimeters, as the official code
#      builds them. Used here.
#   0: AF detections + NIFC perimeters, as the paper's text describes.
# Test fires keep each fire's rule from the 2021 fire list (as in the paper and official code).
# Processing skips fires already in the window repo, so delete train/ and val/ archives built with
# the other rule (and the old data.zip) before rebuilding; each archive records its label_sel.
TRAIN_VAL_LABEL_SEL = 1

# Hugging Face dataset repos ("owner/name"). Change any of them to use your own.
SRC_REPO = "SamuelWu318/ts-satfire"            # raw GeoTIFFs + ROI CSVs (read)
TESSERA_REPO = "SamuelWu318/ts-tesserafire"    # per-fire TESSERA + tessera.zip
WINDOW_REPO = None                             # None: SamuelWu318/ts-satfire-processed-<TS_LENGTH>-<TS_INTERVAL>

WORK_DIR = "/content"  # every download, cache and temporary file stays on this VM

# Two fires with prior-year TESSERA for the timing test: a 2018 train fire and a 2021 test fire.
TEST_FIRES = ["21890003", "US_2021_MT4714310953420211004"]


In [ ]:
REPO_DIR = f"{WORK_DIR}/fire-research-and-innovation"
BRANCH = "explain"  # branch with the firepred package; "main" once it is merged

!git clone -b {BRANCH} https://github.com/SamuelWu318/fire-research-and-innovation.git {REPO_DIR} 2>/dev/null || (git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} checkout {BRANCH} && git -C {REPO_DIR} pull origin {BRANCH})
!git -C {REPO_DIR} log --oneline -1
%cd {REPO_DIR}
!pip install -q -U huggingface_hub imagecodecs rasterio pyyaml pandas "geotessera>=0.11"

In [ ]:
from firepred.data import tessera_master, windows_master

# Every path this notebook uses, and the caches pinned under WORK_DIR before huggingface_hub loads.
work = windows_master.workspace(WORK_DIR, TS_LENGTH, TS_INTERVAL, use_tessera=USE_TESSERA, window_repo=WINDOW_REPO)
windows_master.pin_caches(work.cache_dir)
TOKEN = windows_master.read_secret("HF_TOKEN")          # read
TOKEN_ALL = windows_master.read_secret("HF_TOKEN_ALL")  # read + write

print("SRC_REPO:    ", SRC_REPO)
print("WINDOW_REPO: ", work.window_repo, f"({TS_LENGTH} days every {TS_INTERVAL})")
print("TESSERA_REPO:", TESSERA_REPO)
print("DATA_ROOT:   ", work.data_root)


# DATA GRAB
Streams raw GeoTIFFs from `SRC_REPO` into one windowed NPZ per fire in `WINDOW_REPO` (`train/`, `val/`, `test/`), using `TS_LENGTH` input days every `TS_INTERVAL` days. Train/val labels follow `TRAIN_VAL_LABEL_SEL` (1, used here: active-fire detections only, as the official code; 0: detections + NIFC perimeters, as the paper's text). Each archive records the rule that built it; delete archives built with the other rule before rebuilding, since existing fires are skipped. The repo records its settings in `window_config.json` and refuses windows with other settings. Only run if the processed dataset is empty; fires already in `WINDOW_REPO` are skipped.

In [ ]:
# Debugging: add only_fires=['20568194', '20701026'] to build just those fires.
report = windows_master.process_windows(
    TOKEN_ALL, src_repo=SRC_REPO, window_repo=work.window_repo, length=TS_LENGTH, interval=TS_INTERVAL,
    work_dir=WORK_DIR, roi_dir=work.roi_dir, splits=("test", "train", "val"),
    batch_size=3, train_val_label_sel=TRAIN_VAL_LABEL_SEL,
)


# TESSERA
For each fire, reads the crop georeference from its first and last VIIRS_Day GeoTIFF in `SRC_REPO`, streams the prior-year TESSERA embeddings over that crop into RAM, averages them onto the 256x256 grid, and uploads `tessera/p<id>.npz` (embedding + coverage + metadata) to `TESSERA_REPO` after each fire. 2017 fires have no prior-year TESSERA and are skipped.

1. **Timing test** on `TEST_FIRES`; check the `built in N min` lines and the coverage printout.
2. **Full run**; fires already uploaded are skipped, so rerun the cell after a disconnect to resume.
3. **Bundle** `tessera/` plus the drop list into `tessera.zip` in `TESSERA_REPO`; it warns about any kept fire still missing.

In [ ]:
# 1) timing test
# test_report = tessera_master.process_tessera(TOKEN_ALL, src_repo=SRC_REPO, tessera_repo=TESSERA_REPO,
#                                              work_dir=WORK_DIR, roi_dir=work.roi_dir, only_fires=TEST_FIRES)


In [ ]:
# 2) all fires (resumable)
# tessera_report = tessera_master.process_tessera(TOKEN_ALL, src_repo=SRC_REPO, tessera_repo=TESSERA_REPO,
#                                                 work_dir=WORK_DIR, roi_dir=work.roi_dir)


In [ ]:
# 3) tessera.zip (warns about kept fires still missing a TESSERA file)
# zip_report = tessera_master.upload_tessera_zip(TOKEN_ALL, src_repo=SRC_REPO, tessera_repo=TESSERA_REPO,
#                                                work_dir=WORK_DIR, roi_dir=work.roi_dir)


# DATA LOADING
Rebuilds `data.zip` in `WINDOW_REPO` from the windowed NPZs. Only needed after DATA GRAB changed them.

In [ ]:
# Downloads every archive of the window repo and uploads them back as data.zip.
# upload = windows_master.rebuild_data_zip(TOKEN_ALL, window_repo=work.window_repo, length=TS_LENGTH,
#                                          interval=TS_INTERVAL, src_repo=SRC_REPO, work_dir=WORK_DIR,
#                                          roi_dir=work.roi_dir)


# PULL ZIP
For training in any Colab: pulls `data.zip` from `WINDOW_REPO` into `DATA_ROOT`. In TESSERA mode it also unpacks `tessera.zip` from `TESSERA_REPO` into `DATA_ROOT/tessera` and removes the 2017 fires' window archives.

In [ ]:
# windows_master.pull_data_zip(work.data_root, token=TOKEN, repo_id=work.window_repo)
# if USE_TESSERA:
#     pulled = tessera_master.pull_tessera_zip(work.data_root, token=TOKEN, tessera_repo=TESSERA_REPO)
